# React — Custom Hooks

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> This topic is playground work. A custom Hook is a function that calls other Hooks, so it
> needs a real component to live in.

## LESSON 62 — Extracting a custom Hook

Topic 13 told you what a custom Hook is in one sentence and moved on. Two topics have since
left you something worth extracting: topic 15's fetch-in-an-Effect and topic 20's search params
were both written out longhand, deliberately, so that this lesson has real duplication to
remove.

### What one is

A custom Hook is **a function whose name starts with `use` and which calls other Hooks**. That
is the entire definition — there is no new API here, no import, nothing to install.

```jsx
function useToggle(initial = false) {
  const [on, setOn] = useState(initial);
  const toggle = () => setOn((current) => !current);
  return [on, toggle];
}
```

`useToggle` is an ordinary function. It happens to call `useState`, which makes it a Hook, which
means LESSON 37's rules now apply **to it**: it must be called at the top level of a component
or of another custom Hook, never in a condition or a loop.

### The naming rule is not cosmetic

> **Hook names must start with `use` followed by a capital letter,** like `useState` (built-in)
> or `useOnlineStatus` (custom). Hooks may return arbitrary values.

And the reason:

> This convention guarantees that you can always look at a component and know where its state,
> Effects, and other React features might "hide".

That is the payoff. A reader scanning your component can tell, from the names alone, which lines
might hold state or start an Effect. Name it `getToggle` and you have hidden a Hook inside
something that looks like a plain helper — and the lint rules, which match on the prefix, stop
checking it.

### What it should return

Whatever is most convenient to use. React is explicit that there is no rule:

> Hooks may return arbitrary values.

Three shapes cover almost everything:

| return | when |
|---|---|
| a single value | the Hook is a read-only source — `useOnlineStatus()` |
| an array `[value, setter]` | there are exactly two things and the caller will rename them |
| an object `{ query, setQuery, results }` | there are several things, and names help |

Use an array when you expect the caller to rename (like `useState`), and an object once there
are more than two values — `const { text, query } = useSearchBox()` reads better than remembering
a position.

### When to extract — and when not to

The most useful guidance React gives is a warning against over-extracting:

> You don't need to extract a custom Hook for every little duplicated bit of code. Some
> duplication is fine… However, whenever you write an Effect, consider whether it would be
> clearer to also wrap it in a custom Hook.

So the trigger is not "this appears twice". It is:

- **there is an Effect**, and naming it would say what it is for, or
- **the same stateful logic genuinely appears in more than one component**, or
- **a component is hard to read** because plumbing is mixed with what it renders.

And the shape to aim for:

> **Keep custom Hooks focused on concrete high-level use cases.** Avoid creating and using
> custom "lifecycle" Hooks that act as alternatives and convenience wrappers for the
> `useEffect` API itself.

React's own examples of the wrong kind are worth memorising: `useMount(fn)`, `useEffectOnce(fn)`,
`useUpdateEffect(fn)`. Those describe *when React runs things*, which is React's vocabulary, not
your application's. The right kind name a job: `useData(url)`, `useChatRoom(options)`,
`useImpressionLog(eventName)`.

> A good custom Hook makes the calling code more declarative by constraining what it does. For
> example, `useChatRoom(options)` can only connect to the chat room.

### Key Notes

- A custom Hook is a function starting with `use` that calls other Hooks. No new API.
- The `use` prefix is how readers and lint rules know where state and Effects hide.
- Return whatever is convenient — a value, a pair, or an object once there are several things.
- Extract when there is an Effect worth naming or genuine repeated logic. Some duplication is
  fine.
- Name the **job**, not the lifecycle. `useChatRoom`, not `useMount`.

### Example

**In the playground.** No cell — a custom Hook calls other Hooks, and Hooks need a component.

Point `playground/src/App.jsx` at `./experiments/27-custom-hooks.jsx`. Read `useToggle` first:
six lines, no imports beyond `useState`, and nothing about it is new.

### Exercise

**In the playground**, in `27-custom-hooks.jsx`.

1. Read `useToggle`. Rename it to `getToggle` everywhere and reload. Does it still work? Then
   say what you have lost, in terms of what a reader and a linter can know.
2. Change `useToggle` to return an **object** `{ on, toggle }` instead of an array, and update
   both `Panel` and `useSearchBox`. Which reads better here, and why does `useState` itself use
   the array shape?
3. Write `useOnlineStatus()` returning a single boolean, using `navigator.onLine` and the
   `online`/`offline` window events. It needs an Effect with cleanup (LESSON 41). Display it.
4. In a comment: React warns against `useMount(fn)` and `useEffectOnce(fn)`. Explain what is
   wrong with those names using the phrase "concrete high-level use case", and give a better
   name for a Hook whose job is "log a page view when this screen appears".

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

For each, say whether you would extract a custom Hook, and if so what you would call it.

1. Three components each run the same `useEffect` that subscribes to a websocket and cleans up.
2. Two components each have `const [open, setOpen] = useState(false)`.
3. Five components each fetch from a different URL with the same loading/error/empty handling.
4. One component has forty lines of fetch plumbing above eight lines of JSX.
5. Two components each format a date the same way.

Then answer: number 2 and number 4 pull in opposite directions — one is duplication that is not
worth extracting, the other is a single use worth extracting anyway. What distinguishes them?

In [ ]:
// Your code here

## LESSON 63 — Three you will actually write

Three custom Hooks come up often enough to be worth writing once, properly. All three are
short. None of them uses anything you have not already met.

### `useToggle` — the smallest useful one

```jsx
function useToggle(initial = false) {
  const [on, setOn] = useState(initial);
  const toggle = () => setOn((current) => !current);
  return [on, toggle];
}
```

Note the updater form `setOn((current) => !current)` rather than `setOn(!on)` — LESSON 26's
rule. Inside a Hook you have no idea what the calling component does around it, so always take
the safe form.

### `useDebounce` — LESSON 42, extracted

You wrote this in an Effect in topic 14. Extracting it changes nothing about how it works; it
changes what the calling component reads like.

```jsx
function useDebounced(value, delay = 400) {
  const [debounced, setDebounced] = useState(value);

  useEffect(() => {
    const id = setTimeout(() => setDebounced(value), delay);
    return () => clearTimeout(id);        // the cleanup is the whole trick
  }, [value, delay]);

  return debounced;
}
```

Before: a component with a `useState`, a `useEffect`, a `setTimeout`, a `clearTimeout` and two
variables to keep straight. After:

```jsx
const [text, setText] = useState("");
const query = useDebounced(text);
```

That is the case React makes for extracting Effects: the Effect is still there, but the calling
code now says *what* rather than *how*.

### `useLocalStorage` — state that survives a reload

`localStorage` is a browser API you met in the JavaScript course: `getItem`, `setItem`, strings
only, so JSON in and out. The Hook pairs it with state.

```jsx
function useLocalStorage(key, initialValue) {
  const [value, setValue] = useState(() => {
    const stored = localStorage.getItem(key);
    return stored === null ? initialValue : JSON.parse(stored);
  });

  useEffect(() => {
    localStorage.setItem(key, JSON.stringify(value));
  }, [key, value]);

  return [value, setValue];
}
```

Three details carry all of this lesson's weight:

1. **The lazy initialiser** `useState(() => …)` — LESSON 26. Passing `useState(JSON.parse(…))`
   would read and parse storage on *every* render and throw the result away.
2. **`stored === null`**, not `if (stored)`. `getItem` returns `null` for a missing key, and
   `null` is the only signal of absence there is. With `JSON.stringify` in front of it you would
   get away with a truthiness check by luck — `JSON.stringify(0)` is the string `"0"` and
   `JSON.stringify("")` is `'""'`, both truthy. But store a raw string instead, such as a draft
   the user cleared, and `getItem` hands back `""`, which `if (stored)` reads as "nothing saved".
   Comparing to `null` says what you actually mean and keeps saying it when the stored shape
   changes.
3. **`JSON.parse` can throw** on data another tab or an older version of your app wrote. The
   Hook above trusts storage; the exercise asks you to fix that.

The returned `setValue` is React's own setter, so `setValue(v => v + 1)` works exactly as it
does with `useState`. The Effect writes whatever lands in state, and it runs after the render
commits — never during it.

### Key Notes

- Extracting an Effect does not change its behaviour; it changes what the caller reads like.
- Inside a Hook, always use the updater form — you do not control the caller.
- `useState(() => …)` for anything expensive to compute, such as reading and parsing storage.
- `getItem` returns `null` for a missing key. Compare to `null`, not truthiness.

### Example

**Runnable — plain JS.** No React here, just the storage round-trip the Hook depends on, so you
can see exactly what comes back and in what type. Deno gives you a real `localStorage`.

In [ ]:
// L63 — what localStorage actually returns

const l63Key = "l63-demo";

localStorage.setItem(l63Key, JSON.stringify({ theme: "dark", count: 0 }));

const l63Raw = localStorage.getItem(l63Key);
console.log("type of what comes back:", typeof l63Raw);
console.log("raw:", l63Raw);
console.log("parsed:", JSON.parse(l63Raw));

// a missing key
console.log("missing key returns:", localStorage.getItem("l63-never-written"));

// is `if (stored)` ever actually wrong? measure it rather than assume.
// through JSON, no: stringify never produces a falsy string.
localStorage.setItem("l63-zero", JSON.stringify(0));
localStorage.setItem("l63-empty", JSON.stringify(""));
console.log("JSON 0  ->", localStorage.getItem("l63-zero"), "| truthy?", Boolean(localStorage.getItem("l63-zero")));
console.log('JSON "" ->', localStorage.getItem("l63-empty"), "| truthy?", Boolean(localStorage.getItem("l63-empty")));

// stored as a raw string — a draft the user cleared — and it IS wrong:
localStorage.setItem("l63-draft", "");
const l63Draft = localStorage.getItem("l63-draft");
console.log("raw empty draft ->", JSON.stringify(l63Draft), "| truthy?", Boolean(l63Draft));
console.log("  if (stored) says:", l63Draft ? "saved" : "NOTHING SAVED — wrong, it was saved");
console.log("  stored === null says:", l63Draft === null ? "nothing saved" : "saved — correct");

["l63-zero", "l63-empty", "l63-draft"].forEach((k) => localStorage.removeItem(k));
localStorage.removeItem(l63Key);

### Exercise

Parts 1 and 2 are **runnable — plain JS**. Part 3 is **in the playground**.

1. Write `l63ReadJSON(key, fallback)`: return the parsed value, or `fallback` if the key is
   missing **or** if the stored text is not valid JSON. Use `try`/`catch`. Test it three ways —
   a good value, a missing key, and a key you deliberately set to the string `"{oops"`.
2. Write `l63WriteJSON(key, value)` and use both to round-trip an array of three objects.
   Confirm the value that comes back is deep-equal to what you put in.
3. **In the playground**, in `27-custom-hooks.jsx`: add `useLocalStorage` as written above, but
   using your `l63ReadJSON` logic for the initialiser. Use it for the `showTips` toggle so the
   panel state survives a page reload. Reload and confirm.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** A subtle one about the lazy initialiser.

Write two functions, `l63Eager()` and `l63Lazy()`, that both fake what `useState` does with its
argument: `l63Eager` takes a value, `l63Lazy` takes a function and calls it. Give each a
`readStorage()` that logs `"reading storage"` and returns a parsed object. Now call each one
**three times**, the way three renders of a component would.

Count the log lines. Then answer in a comment: when React re-renders a component,
`useState(readStorage())` and `useState(readStorage)` both give the same state value — so what
exactly is the difference, and why does one of them still cost something on every render?

In [ ]:
// Your code here

## LESSON 64 — Composing Hooks, and the trap

A custom Hook can call another custom Hook. That is not a special feature — `useToggle` calls
`useState`, and `useState` is a Hook, so composition is what you have been doing since LESSON
62's first example.

```jsx
function useSearchBox(initial = "") {
  const [text, setText] = useState(initial);
  const query = useDebounced(text);          // a custom Hook, inside a custom Hook
  const [showTips, toggleTips] = useToggle(false);

  return { text, setText, query, showTips, toggleTips };
}
```

The calling component is now one line: `const search = useSearchBox()`. Everything about
timers, cleanup and two pieces of state moved somewhere it can be named.

The rules do not change when you nest. `useDebounced` must still be called at the top level of
`useSearchBox` — no condition, no loop, no early return above it. LESSON 37 applies to every
function in the chain.

### The trap

This is the single most common misunderstanding about custom Hooks, and React states it flatly:

> **Custom Hooks let you share stateful logic, not state itself.**

Two components calling `useToggle()` do **not** share a toggle. They each get their own.

> Each call to a Hook is completely independent from every other call to the same Hook.

Say it in terms of ordinary functions and it stops being mysterious: `useToggle` is a function
that calls `useState`, and every call to `useState` creates a separate piece of state in
whichever component is rendering. Two calls, two states — exactly as if you had written the
`useState` line twice by hand, because effectively you did.

So this does **not** work:

```jsx
// WRONG — expecting the sidebar to open when the header button is pressed
function Header()  { const [open, toggle] = useToggle(); /* … */ }
function Sidebar() { const [open, toggle] = useToggle(); /* … */ }
```

Two independent `open` values. Pressing the header's button leaves the sidebar shut.

### When you really do want shared state

Nothing new is needed — it is what topics 09 and 18 already taught:

| what you want | what you use |
|---|---|
| the same *logic* in many components | a custom Hook |
| the same *state* in two siblings | lift it up (LESSON 29) and pass it down |
| the same state across a whole subtree | Context (LESSON 56) |

The combination is common and worth recognising: call the custom Hook **once**, in the
provider, and put what it returns into a Context. Now the logic lives in one named place and
the state has exactly one owner.

### Key Notes

- Hooks compose: a custom Hook may call custom Hooks, under the same Rules of Hooks.
- Custom Hooks share **logic, not state**. Every call is completely independent.
- Two components calling the same Hook get two separate states — never a channel between them.
- For shared state, lift it up or use Context. A custom Hook called inside the provider gives
  you both.

### Example

**Runnable — plain JS.** The trap, without React. A closure factory behaves exactly the way
your Hook does: calling it twice gives two independent boxes. If that is obvious here, it is
the same fact in React.

In [ ]:
// L64 — two calls, two states

function l64MakeToggle(initial = false) {
  let on = initial;                       // stands in for the component's state slot
  return {
    read: () => on,
    toggle: () => { on = !on; },
  };
}

const l64Header = l64MakeToggle();
const l64Sidebar = l64MakeToggle();

console.log("start   — header:", l64Header.read(), "sidebar:", l64Sidebar.read());

l64Header.toggle();

console.log("toggled — header:", l64Header.read(), "sidebar:", l64Sidebar.read());
console.log("shared?", l64Header.read() === l64Sidebar.read());

// the same function, called once and SHARED instead
const l64Shared = l64MakeToggle();
const l64A = l64Shared;
const l64B = l64Shared;

l64A.toggle();
console.log("one instance, two readers — a:", l64A.read(), "b:", l64B.read());

### Exercise

Part 1 is **runnable — plain JS**. Parts 2 and 3 are **in the playground**.

1. Extend `l64MakeToggle` into `l64MakeCounter(start)` with `read`, `increment` and `reset`.
   Create two counters, increment one four times and the other once, and print both. Then
   create one counter and hand it to two different functions that each increment it, and print
   the result. In a comment, name which of the two arrangements matches a custom Hook.
2. **In the playground**, in `27-custom-hooks.jsx`: the two `<Panel />` components already prove
   the trap. Add a third `<Panel title="third" />`, open the first, and confirm the other two
   are unaffected.
3. Now make the header and the panels share one open/closed value, without deleting `useToggle`.
   Call `useToggle` **once** in `Experiment27` and pass `open` and `toggle` down as props. Both
   panels must now open and close together. Keep the third panel on its own `useToggle` so you
   can see both arrangements side by side.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** A design question with a small model.

A colleague writes `useAppState()`, a custom Hook that holds the logged-in user, the theme, the
shopping cart and the current search query, and every component in the app calls it.

1. Model it: write `l64MakeAppState()` returning an object with all four values. Create two
   instances, change the theme on one, and print both.
2. In a comment, answer three things: what does a component that only needs the theme get from
   this Hook that it does not want? Why does calling it in two components fail to share the
   cart? And which two words from LESSON 62 — about what a good custom Hook is focused on —
   describe the fix?

In [ ]:
// Your code here